In [141]:
import numpy as np
import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq
from sklearn.manifold import TSNE
import scienceplots
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


from bscarlos.settings import PROCESSED_KISPI_DATA_FOLDER

plt.style.use('science')

# Preproccess EEG Data

In [120]:
file_path = PROCESSED_KISPI_DATA_FOLDER / Path("SE021_a1_filt_merged.parquet")

table = pq.read_table(file_path)

df = table.to_pandas()

print(df.head())

     Fp1-F7     F7-T3     T3-T5    Fp2-F8     F8-T4     T4-T6  ground_truth
0  0.124338  0.000000 -0.165783  0.331567  0.306699 -0.099470             0
1  0.248675 -0.132627  0.447615  2.851475  1.632967 -0.886941             0
2  0.314988 -0.215518  1.152195  5.213888  2.851475 -1.657834             0
3  0.223808 -0.165783  2.005979  7.269603  3.837886 -2.378992             0
4 -0.033157  0.041446  3.066993  8.885991  4.525887 -3.025547             0


## Remove Ground Truth Labels

In [121]:
eeg_data = df.drop('ground_truth', axis=1)
eeg_data

,Fp1-F7,F7-T3,T3-T5,Fp2-F8,F8-T4,T4-T6
0,0.124338,0.000000,-0.165783,0.331567,0.306699,-0.099470
1,0.248675,-0.132627,0.447615,2.851475,1.632967,-0.886941
2,0.314988,-0.215518,1.152195,5.213888,2.851475,-1.657834
3,0.223808,-0.165783,2.005979,7.269603,3.837886,-2.378992
4,-0.033157,0.041446,3.066993,8.885991,4.525887,-3.025547
...,...,...,...,...,...,...
368635,7.153554,0.613399,-2.834896,-2.851475,4.103140,5.843864
368636,5.230467,0.447615,-2.138606,-2.453595,3.224487,4.227477
368637,3.249355,0.265253,-1.434026,-1.807039,2.221498,2.611089
368638,1.243376,0.074603,-0.729447,-0.978122,1.135616,0.978122


In [122]:
eeg_data = eeg_data.values

## Normalize

Since VAE uses a sigmoid activation in the decoder, normalize values between 0 and 1

In [123]:
eeg_data_min = eeg_data.min()
print(eeg_data_min)
eeg_data_max = eeg_data.max()
print(eeg_data_max)
eeg_data = (eeg_data - eeg_data_min) / (eeg_data_max - eeg_data_min)

-203.83899
212.71669


## Segment
Segment EEG data into 2-second windows (512 samples at 256Hz)

$$window size=512(samples)×6(channels)=3072(features)$$

In [124]:
window_size = 512
num_windows = eeg_data.shape[0] // window_size
print(num_windows)

720


## Reshape

reshaped_data.shape=(num_windows,3072)

In [125]:
reshaped_data = eeg_data[:num_windows * window_size]  # Trim excess samples
reshaped_data = reshaped_data.reshape(-1, window_size * 6)  # Shape: (num_windows, 3072)
print(reshaped_data.shape)

(720, 3072)


# Hyperparameters

In [126]:
input_dim = 3072  # 512 samples * 7 channels
output_dim = 3072  # Reconstruction matches input
latent_dim = 32 # {2^n}, 5 <= n <= 7 [32, 64 , 128]
batch_size = 64 # typically 32-64 for EEG data
lr = 1e-3 # 1e-4, 1e-5
num_epochs = 10
beta = 0.1 # 0.1 - 1


## Create Pytorch Dataset

In [127]:
class EEGDataset(Dataset):
    def __init__(self, data):
        self.data = torch.tensor(data, dtype=torch.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], 0  # Dummy label for VAE

# Instantiate the dataset and DataLoader
dataset = EEGDataset(reshaped_data)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Number of batches: {len(train_loader)}")


Number of batches: 12


# Encoder
  The encoder network takes in the data `input_dim`, processes it through a few fully connected layers, and outputs the mean ($mu$) and log-variance (log_var) of the latent distribution. $$input\_dim=512×7=3584$$

In [128]:
# Define the Encoder network
class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Encoder, self).__init__()
        # Define a simple feed-forward network for the encoder
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3_mu = nn.Linear(256, latent_dim)  # Mean of the latent variable
        self.fc3_logvar = nn.Linear(256, latent_dim)  # Log variance of the latent variable

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        mu = self.fc3_mu(x)
        logvar = self.fc3_logvar(x)
        return mu, logvar

# Decoder

In [129]:
# Define the Decoder network
class Decoder(nn.Module):
    def __init__(self, latent_dim, output_dim):
        super(Decoder, self).__init__()
        # Define a simple feed-forward network for the decoder
        self.fc1 = nn.Linear(latent_dim, 256)
        self.fc2 = nn.Linear(256, 512)
        self.fc3 = nn.Linear(512, output_dim)

    def forward(self, z):
        z = F.relu(self.fc1(z))
        z = F.relu(self.fc2(z))
        reconstruction = torch.sigmoid(self.fc3(z))  # Sigmoid for binary outputs
        return reconstruction

# Reparameterization Trick

In [130]:
# Reparameterization trick
def reparameterize(mu, logvar):
    std = torch.exp(0.5*logvar)
    eps = torch.randn_like(std)
    z = mu + eps*std
    return z

# VAE Model

In [131]:
# VAE Model
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim, output_dim):
        super(VAE, self).__init__()
        self.encoder = Encoder(input_dim, latent_dim)
        self.decoder = Decoder(latent_dim, output_dim)

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = reparameterize(mu, logvar)
        reconstructed_x = self.decoder(z)
        return reconstructed_x, mu, logvar, z # Retrun latent variables (z)

    def loss_function(self, recon_x, x, mu, logvar, beta=0.1):
        # Reconstruction loss (binary cross-entropy)
        #BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
        # Reconstruction loss (MSE for continuous data)
        #MSE = F.mse_loss(recon_x, x, reduction='sum')
        # L1 Reconstruction loss (absolute error)
        recon_loss = torch.sum(torch.abs(recon_x - x))

        # KL divergence loss
        # Compute the KL divergence between the learned distribution and the prior (N(0, I))
        # Use the formula: KL(q(z|x) || p(z)) = 0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
        # where mu is the mean and logvar is the log variance
        # sigma^2 = exp(logvar)
        # Here we calculate the negative log-likelihood in a variational autoencoder
        # for an isotropic Gaussian distribution
        KL = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

        # The total loss is the sum of the reconstruction loss and KL divergence
        total_loss = recon_loss + beta * KL # + BCA, + MSE
        return total_loss 

# Hyperparameters
input_dim = input_dim
latent_dim = latent_dim
output_dim = output_dim

# Create the VAE model
model = VAE(input_dim=input_dim, latent_dim=latent_dim, output_dim=output_dim)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=lr)


# Train

In [132]:
# Training loop + monitor both loss components
def train(model, train_loader, optimizer, num_epochs=num_epochs):
    model.train()
    for epoch in range(num_epochs):
        train_loss, recon_loss, kl_loss = 0, 0, 0
        for batch_idx, (data, _) in enumerate(train_loader):
            optimizer.zero_grad()
            recon_batch, mu, logvar, _ = model(data)
            loss = model.loss_function(recon_batch, data, mu, logvar)

            # Calculate reconstruction and KL divergence loss separately
            MSE = F.mse_loss(recon_batch, data, reduction='sum')
            KL = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

            loss.backward()
            train_loss += loss.item()
            recon_loss += MSE.item()
            kl_loss += KL.item()
            optimizer.step()

        print(f'Epoch {epoch+1}, Total Loss: {train_loss / len(train_loader.dataset):.4f}, '
              f'Recon Loss: {recon_loss / len(train_loader.dataset):.4f}, '
              f'KL Divergence: {kl_loss / len(train_loader.dataset):.4f}')

In [133]:
# Train the model
train(model, train_loader, optimizer)

Epoch 1, Total Loss: 102.9673, Recon Loss: 7.2246, KL Divergence: 6.3575
Epoch 2, Total Loss: 98.2678, Recon Loss: 6.8488, KL Divergence: 1.5081
Epoch 3, Total Loss: 97.6054, Recon Loss: 6.8015, KL Divergence: 0.7165
Epoch 4, Total Loss: 97.3574, Recon Loss: 6.7839, KL Divergence: 0.3854
Epoch 5, Total Loss: 97.1787, Recon Loss: 6.7710, KL Divergence: 0.2736
Epoch 6, Total Loss: 97.0775, Recon Loss: 6.7623, KL Divergence: 0.1935
Epoch 7, Total Loss: 97.0085, Recon Loss: 6.7591, KL Divergence: 0.1208
Epoch 8, Total Loss: 96.9607, Recon Loss: 6.7556, KL Divergence: 0.0827
Epoch 9, Total Loss: 96.9532, Recon Loss: 6.7557, KL Divergence: 0.0569
Epoch 10, Total Loss: 96.9181, Recon Loss: 6.7520, KL Divergence: 0.0369


# Plot Latent Space Variables (z)

In [134]:
def tsne_latent_space(model, train_loader):
    model.eval()
    all_latents = []
    all_labels = []  # Store labels if available, can be useful for coloring the plot

    with torch.no_grad():  # No need to track gradients during inference
        for data, labels in train_loader:
            recon_batch, mu, logvar, z = model(data)  # Get the latent variable z
            all_latents.append(z.cpu().numpy())  # Move data to CPU and collect
            all_labels.append(labels.cpu().numpy())  # Collect labels (if necessary)

    all_latents = np.concatenate(all_latents, axis=0)  # Concatenate the latent variables
    all_labels = np.concatenate(all_labels, axis=0) 

    # Apply t-SNE
    tsne = TSNE(n_components=2, random_state=42)
    latent_2d = tsne.fit_transform(all_latents) 

    # Plot the latent space
    plt.figure(figsize=(8, 6))
    plt.scatter(latent_2d[:, 0], latent_2d[:, 1], c=all_labels, cmap='jet', alpha=0.5)
    plt.title("Latent Space Visualization with t-SNE")
    plt.xlabel("t-SNE Component 1")
    plt.ylabel("t-SNE Component 2")
    plt.colorbar()
    plt.show()


In [ ]:
#tsne_latent_space(model, train_loader)

In [135]:
def tsne_latent_space_nolabels(model, train_loader):
    model.eval()  # Set the model to evaluation mode
    all_latents = []

    with torch.no_grad():  # No need to track gradients during inference
        for data, _ in train_loader:  # We don't need labels here
            recon_batch, mu, logvar, z = model(data)  # Get the latent variable z
            all_latents.append(z.cpu().numpy())  # Collect latent variables

    all_latents = np.concatenate(all_latents, axis=0)  # Concatenate the latent variables

    # Apply t-SNE to reduce to 2D for visualization
    tsne = TSNE(n_components=2, random_state=42 #,metric='precomputed', init='random')
    latent_2d = tsne.fit_transform(all_latents)  # Project to 2D space

    # Plot the latent space
    plt.figure(figsize=(8, 6))
    plt.scatter(latent_2d[:, 0], latent_2d[:, 1], alpha=0.5)
    plt.title("Latent Space Visualization with t-SNE")
    plt.xlabel("t-SNE Component 1")
    plt.ylabel("t-SNE Component 2")
    plt.show()


In [142]:
tsne_latent_space_nolabels(model, train_loader)

RuntimeError: latex was not able to process the following string:
b'lp'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error --output-directory=tmproauwpjp 45b35e7c45bf1f94c12d11a2e5717b88.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.25 (MiKTeX 23.4) (preloaded format=latex.fmt)
 restricted \write18 enabled.
entering extended mode
(45b35e7c45bf1f94c12d11a2e5717b88.tex
LaTeX2e <2022-11-01> patch level 1
L3 programming layer <2023-03-30>
(C:\Users\c_arz\AppData\Local\Programs\MiKTeX\tex/latex/base\article.cls
Document Class: article 2022/07/02 v1.4n Standard LaTeX document class
(C:\Users\c_arz\AppData\Local\Programs\MiKTeX\tex/latex/base\size10.clo))
(C:\Users\c_arz\AppData\Local\Programs\MiKTeX\tex/latex/type1cm\type1cm.sty)
(C:\Users\c_arz\AppData\Local\Programs\MiKTeX\tex/latex/cm-super\type1ec.sty
(C:\Users\c_arz\AppData\Local\Programs\MiKTeX\tex/latex/base\t1cmr.fd))
(C:\Users\c_arz\AppData\Local\Programs\MiKTeX\tex/l

<Figure size 800x600 with 1 Axes>